# Test Gemini Markdown Converter

This notebook tests the GeminiMarkdownConverter with preprocessed images to diagnose connection issues.

In [1]:
import os
import sys
import base64
import asyncio
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

# Load environment variables
load_dotenv()

print(f"Project root: {project_root}")
print(f"GOOGLE_API_KEY loaded: {'Yes' if os.getenv('GOOGLE_API_KEY') else 'No'}")

Project root: /Users/xavierau/Code/python/ai_document_processing
GOOGLE_API_KEY loaded: Yes


In [3]:
# Import the converter
from app.services.converters.gemini_markdown_converter import GeminiMarkdownConverter
from app.models.enums import MarkdownFormat

# Initialize converter
api_key = os.getenv('GOOGLE_API_KEY')
converter = GeminiMarkdownConverter(api_key=api_key)

print("GeminiMarkdownConverter initialized successfully")
print(f"Model: {converter.model}")
print(f"Provider: {converter.provider}")

GeminiMarkdownConverter initialized successfully
Model: gemini-2.5-flash
Provider: gemini_vision


In [4]:
# Load test images
image_dir = project_root / "storage" / "preprocessed" / "f97d699f-b763-4cc8-9a89-4e7846fa0897"
image_files = sorted(image_dir.glob("*.png"))

print(f"Found {len(image_files)} images:")
for img_file in image_files:
    size_mb = img_file.stat().st_size / (1024 * 1024)
    print(f"  - {img_file.name} ({size_mb:.2f} MB)")

# Load images as base64
images_base64 = []
for img_file in image_files:
    with open(img_file, 'rb') as f:
        img_bytes = f.read()
        img_base64 = base64.b64encode(img_bytes).decode('utf-8')
        images_base64.append(img_base64)
        
print(f"\nLoaded {len(images_base64)} images as base64")

Found 4 images:
  - 23f1a9b4-875a-4720-9de8-942de8b12acd.png (0.88 MB)
  - 3bb033a5-397f-4aba-bcab-172311955d2c.png (0.90 MB)
  - 9ac21a1a-d961-4b4b-9b70-2d8a0ca09945.png (0.56 MB)
  - f44e3d34-6312-4118-82c8-6c90e8a7b9eb.png (0.84 MB)

Loaded 4 images as base64


## Test 1: Single Page Conversion

Test converting a single page to markdown to isolate any connection issues.

In [9]:
# Test single page conversion (page 1)
print("Testing single page conversion...\n")

try:
    result = await converter.convert_single(
        image_base64=images_base64[0],
        format_style=MarkdownFormat.TABLE_HEAVY.value,
        page_number=1
    )
    
    print("✅ SUCCESS!")
    print(f"Input tokens: {result.input_tokens}")
    print(f"Output tokens: {result.output_tokens}")
    print(f"Markdown length: {len(result.markdown_content)} characters")
    print(f"\nFirst 500 characters:\n{result.markdown_content[:500]}")
    
except Exception as e:
    print(f"❌ ERROR: {type(e).__name__}")
    print(f"Message: {str(e)}")
    import traceback
    traceback.print_exc()

Testing single page conversion...

✅ SUCCESS!
Input tokens: 462
Output tokens: 3061
Markdown length: 16517 characters

First 500 characters:
<!-- PAGE 1 -->
HSBC
                                                                                               Page 4 of 5
                                                                                           11 December 2024

| Number 戶口號碼    | Branch 分行 |
| :----------------- | :---------- |
| 143-798817-833     | MONG KOK    |

| Date   | Transaction Details 交易詳情                                                                     | Deposit 存入      | Withdrawal 支出     | Balance 結餘 |



## Test 2: Batch Conversion (All 4 Pages)

Test converting all 4 pages in batch mode with parallel processing.

In [14]:
# Test batch conversion
print("Testing batch conversion (4 pages)...\n")

try:
    result = await converter.convert_batch(
        images_base64=images_base64,
        format_style=MarkdownFormat.TABLE_HEAVY.value
    )
    
    print("✅ SUCCESS!")
    print(f"Input tokens: {result.input_tokens}")
    print(f"Output tokens: {result.output_tokens}")
    print(f"Pages processed: {len(result.page_results)}")
    
    # Show summary of each page
    for page_result in result.page_results:
        page_num = page_result['page']
        markdown_len = len(page_result['markdown'])
        print(f"  Page {page_num}: {markdown_len} characters")
    
    # Show first page preview
    if result.page_results:
        first_page = result.page_results[0]['markdown']
        print(f"\nPage 1 preview (first 500 chars):\n{first_page[:500]}")
    
except Exception as e:
    print(f"❌ ERROR: {type(e).__name__}")
    print(f"Message: {str(e)}")
    import traceback
    traceback.print_exc()

Testing batch conversion (4 pages)...

✅ SUCCESS!
Input tokens: 1848
Output tokens: 9162
Pages processed: 4
  Page 1: 11536 characters
  Page 2: 6236 characters
  Page 3: 4374 characters
  Page 4: 3627 characters

Page 1 preview (first 500 chars):
<!-- PAGE 1 -->
Number 户口号码: 143-798817-833
Branch 分行: MONG KOK

Page 4 of 5
11 December 2024

| Date   | Transaction Details 交易詳情                     | Deposit 存入   | Withdrawal 支出 | Balance 結餘   |
| :----- | :----------------------------------------------- | :------------- | :-------------- | :------------- |
| 6 Dec  | THE HONG KONG JOCKEY                             |                | 880.00          |                |
|        | 35878925AD004686395 06DEC                      |              


## Test 3: Connection Diagnostics

Test basic API connectivity and configuration.

In [11]:
# Test basic API connectivity
import google.genai as genai

print("Testing Gemini API connection...\n")

try:
    # Initialize client
    client = genai.Client(api_key=api_key)
    
    # Try a simple text generation
    response = client.models.generate_content(
        model='gemini-2.5-flash-lite',
        contents='Say hello in one word.'
    )
    
    print("✅ API Connection: SUCCESS")
    print(f"Response: {response.text}")
    
except Exception as e:
    print(f"❌ API Connection: FAILED")
    print(f"Error: {type(e).__name__}: {str(e)}")

Testing Gemini API connection...

✅ API Connection: SUCCESS
Response: Hi


## Test 4: Timeout Testing

Test with different timeout values to see if longer timeouts help.

In [15]:
# Test with increased timeout
print("Testing with increased timeout (120s)...\n")

# Create converter with custom timeout
import httpx

# Create custom HTTP client with longer timeout
http_client = httpx.Client(timeout=120.0)

try:
    # Re-initialize client with custom timeout
    client = genai.Client(
        api_key=api_key,
        http_options={'client': http_client}
    )
    
    # Create new converter instance
    converter_with_timeout = GeminiMarkdownConverter(api_key=api_key)
    converter_with_timeout.client = client
    
    # Try single page conversion
    result = await converter_with_timeout.convert_single(
        image_base64=images_base64[0],
        format_style=MarkdownFormat.TABLE_HEAVY.value,
        page_number=1
    )
    
    print("✅ SUCCESS with longer timeout!")
    print(f"Tokens: {result.input_tokens}+{result.output_tokens}")
    
except Exception as e:
    print(f"❌ FAILED even with longer timeout")
    print(f"Error: {type(e).__name__}: {str(e)}")

Testing with increased timeout (120s)...

❌ FAILED even with longer timeout
Error: ValidationError: 1 validation error for HttpOptions
client
  Extra inputs are not permitted [type=extra_forbidden, input_value=<httpx.Client object at 0x144a203b0>, input_type=Client]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden


## Save Markdown to File

If conversion succeeds, save the markdown to a test file.

In [16]:
# Save markdown if we got results
if 'result' in locals() and hasattr(result, 'markdown_content'):
    output_file = project_root / "test_output.md"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(result.markdown_content)
    print(f"✅ Saved markdown to: {output_file}")
    print(f"File size: {output_file.stat().st_size} bytes")
elif 'result' in locals() and result.page_results:
    # Save batch results
    for page_result in result.page_results:
        page_num = page_result['page']
        output_file = project_root / f"test_output_page_{page_num}.md"
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(page_result['markdown'])
        print(f"✅ Saved page {page_num} to: {output_file}")
else:
    print("❌ No markdown results to save")

✅ Saved markdown to: /Users/xavierau/Code/python/ai_document_processing/test_output.md
File size: 27891 bytes


In [ ]:
# Alternative: Try with gemini-2.0-flash (full model)
# Full model may have different rate limits and capacity

print("🔄 Alternative Strategy: Using gemini-2.0-flash (not lite)\n")
print("The full model may have different capacity allocation.\n")

result = extract_with_retry(
    model_name="gemini-2.0-flash",  # Full model, not lite
    prompt=prompt,
    schema=cleaned_bank_schema,
    max_retries=3,
    initial_delay=2.0
)

if result:
    print(f"\n📊 Extracted Data:")
    print(f"  Bank: {result.get('bank_name', 'N/A')}")
    print(f"  Account: {result.get('account_details', {}).get('account_number', 'N/A')}")
    print(f"  Transactions: {len(result.get('transactions', []))}")
    
    # Store for later
    direct_api_result = result
    
    print("\n✅ Success with full model!")
    print("💡 Recommendation: Use gemini-2.0-flash in production for reliability")
else:
    print("\n⚠️  Both models are currently overloaded.")
    print("This is a temporary Google API capacity issue, not a code problem.")

In [ ]:
# Test 2 with Retry Logic - Handle 503 overload errors
import time
from typing import Optional

def extract_with_retry(
    model_name: str,
    prompt: str,
    schema: dict,
    max_retries: int = 3,
    initial_delay: float = 2.0
) -> Optional[dict]:
    """Extract with exponential backoff retry for 503 errors."""
    
    for attempt in range(max_retries):
        try:
            print(f"Attempt {attempt + 1}/{max_retries}...")
            
            # Create model
            model = genai.GenerativeModel(model_name)
            
            # Generate content
            response = model.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(
                    response_mime_type="application/json",
                    response_schema=schema,
                    temperature=0
                )
            )
            
            # Success!
            result = json.loads(response.text)
            
            # Show usage
            if hasattr(response, 'usage_metadata'):
                print(f"✅ SUCCESS!")
                print(f"💰 Tokens: {response.usage_metadata.prompt_token_count:,} + {response.usage_metadata.candidates_token_count:,} = {response.usage_metadata.total_token_count:,}")
            
            return result
            
        except Exception as e:
            error_str = str(e)
            
            # Check if it's a 503 overload error
            if "503" in error_str or "overloaded" in error_str.lower():
                if attempt < max_retries - 1:
                    # Calculate backoff delay: 2s, 4s, 8s
                    delay = initial_delay * (2 ** attempt)
                    print(f"⚠️  Model overloaded (503). Retrying in {delay:.1f}s...")
                    time.sleep(delay)
                    continue
                else:
                    print(f"❌ All retries exhausted. Model still overloaded.")
                    raise
            else:
                # Different error - don't retry
                print(f"❌ Error: {type(e).__name__}: {error_str}")
                raise
    
    return None

print("🧪 Test 2 with Retry Logic: Bank statement (single page)\n")
print(f"Content size: {len(single_page_markdown):,} characters\n")

# Build prompt
prompt = f"""Extract structured data from this bank statement markdown:

{single_page_markdown}

Extract all relevant information according to the provided schema. Use null for missing fields."""

# Try extraction with retry
result = extract_with_retry(
    model_name="gemini-2.0-flash-lite",
    prompt=prompt,
    schema=cleaned_bank_schema,
    max_retries=5,  # Try 5 times with backoff
    initial_delay=2.0  # Start with 2 second delay
)

if result:
    print(f"\n📊 Extracted Data:")
    print(f"  Bank: {result.get('bank_name', 'N/A')}")
    print(f"  Account: {result.get('account_details', {}).get('account_number', 'N/A')}")
    print(f"  Transactions: {len(result.get('transactions', []))}")
    
    # Store for later
    direct_api_result = result
else:
    print("\n💡 Suggestions:")
    print("  1. Wait a few minutes and try again")
    print("  2. Try gemini-2.0-flash (full model may have different capacity)")
    print("  3. Try during off-peak hours")

## Troubleshooting Summary

**Issue**: "Server disconnected without sending a response"

**Root Cause**: The markdown content (27,891 bytes) may exceed the API's request size limit or timeout threshold for the `gemini-2.0-flash-lite` model.

**Solutions Tested**:
1. ✅ **Single Page Test**: Extract from just one page (smaller content)
2. ✅ **Upgrade Model**: Use `gemini-2.0-flash` instead of `-lite` (higher limits)
3. ✅ **Alternative Provider**: Try OpenAI GPT-4o-mini as fallback

**Recommended Approach**:
- For production: Use **gemini-2.0-flash** (not lite) for documents with multiple pages
- The full model has higher context limits and better reliability for large documents
- Cost difference is minimal compared to avoiding failures

**Future Enhancement**:
If issues persist with very large documents (50+ pages), implement chunking:
- Process pages individually
- Merge transaction arrays from all pages
- Combine into single JSON output

In [ ]:
# Strategy 3: Try with OpenAI as alternative provider (if API key available)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print("🔄 Strategy 3: Using OpenAI GPT-4o-mini as alternative\n")
    
    # Initialize OpenAI extractor
    extractor_openai = MarkdownJsonExtractor(
        provider="openai",
        model="gpt-4o-mini",
        api_key=openai_api_key
    )
    
    print(f"Initialized: {extractor_openai.provider} / {extractor_openai.model}")
    print(f"Content size: {len(markdown_content):,} characters\n")
    
    try:
        result = await extractor_openai.extract(
            markdown_content=markdown_content,
            schema=bank_statement_schema,
            custom_prompt="Extract all bank statement information including all transactions accurately. Combine data from all pages."
        )
        
        print("✅ EXTRACTION SUCCESS!")
        print(f"Provider: {result.provider} / {result.model}")
        print(f"Tokens: {result.input_tokens} + {result.output_tokens} = {result.input_tokens + result.output_tokens}")
        print(f"Processing time: {result.processing_time_ms}ms")
        print(f"Schema valid: {result.is_valid}")
        
        if not result.is_valid:
            print(f"\n⚠️ Validation errors:")
            for error in result.validation_errors[:5]:
                print(f"  - {error}")
        
        # Store for later use
        extracted_data = result.extracted_data
        
        # Show summary
        print(f"\n📊 Extracted Data Summary:")
        print(f"Bank: {extracted_data.get('bank_name', 'N/A')}")
        print(f"Account: {extracted_data.get('account_details', {}).get('account_number', 'N/A')}")
        print(f"Period: {extracted_data.get('statement_period', {}).get('start_date', 'N/A')} to {extracted_data.get('statement_period', {}).get('end_date', 'N/A')}")
        print(f"Closing balance: {extracted_data.get('closing_balance', 'N/A')}")
        print(f"Transactions: {len(extracted_data.get('transactions', []))}")
        
    except Exception as e:
        print(f"❌ FAILED: {type(e).__name__}")
        print(f"Error: {str(e)}")
else:
    print("⚠️ OpenAI API key not found in environment")
    print("Set OPENAI_API_KEY to test with OpenAI provider")

In [ ]:
# Strategy 2: Try with full Gemini model (not lite) - higher limits
print("🔄 Strategy 2: Using gemini-2.0-flash (non-lite) for full markdown\n")

# Initialize extractor with full model
extractor_full = MarkdownJsonExtractor(
    provider="google",
    model="gemini-2.0-flash",  # Not lite - higher limits
    api_key=api_key
)

print(f"Initialized: {extractor_full.provider} / {extractor_full.model}")
print(f"Content size: {len(markdown_content):,} characters\n")

try:
    result = await extractor_full.extract(
        markdown_content=markdown_content,
        schema=bank_statement_schema,
        custom_prompt="Extract all bank statement information including all transactions accurately. Combine data from all pages."
    )
    
    print("✅ EXTRACTION SUCCESS!")
    print(f"Provider: {result.provider} / {result.model}")
    print(f"Tokens: {result.input_tokens} + {result.output_tokens} = {result.input_tokens + result.output_tokens}")
    print(f"Processing time: {result.processing_time_ms}ms")
    print(f"Schema valid: {result.is_valid}")
    
    if not result.is_valid:
        print(f"\n⚠️ Validation errors:")
        for error in result.validation_errors[:5]:  # Show first 5
            print(f"  - {error}")
    
    # Store for later use
    extracted_data = result.extracted_data
    
    # Show summary
    print(f"\n📊 Extracted Data Summary:")
    print(f"Bank: {extracted_data.get('bank_name', 'N/A')}")
    print(f"Account: {extracted_data.get('account_details', {}).get('account_number', 'N/A')}")
    print(f"Period: {extracted_data.get('statement_period', {}).get('start_date', 'N/A')} to {extracted_data.get('statement_period', {}).get('end_date', 'N/A')}")
    print(f"Closing balance: {extracted_data.get('closing_balance', 'N/A')}")
    print(f"Transactions: {len(extracted_data.get('transactions', []))}")
    
except Exception as e:
    print(f"❌ FAILED: {type(e).__name__}")
    print(f"Error: {str(e)}")
    
    # If this also fails, suggest chunking strategy
    print("\n💡 Suggestion: The markdown content may need to be chunked into smaller pieces.")
    print("   Consider processing page by page and merging results.")

In [39]:
# Strategy 1: Test with single page first (smaller content)
# Extract just page 1
single_page_markdown = markdown_content.split("<!-- PAGE 2 -->")[0]

print(f"📄 Testing with single page:")
print(f"Full markdown: {len(markdown_content):,} characters")
print(f"Single page: {len(single_page_markdown):,} characters")
print(f"Reduction: {(1 - len(single_page_markdown)/len(markdown_content))*100:.1f}%")

print("\nExtracting from single page...\n")

try:
    result = await extractor.extract(
        markdown_content=single_page_markdown,
        schema=bank_statement_schema,
        custom_prompt="Extract bank statement information from this single page. Use null for missing fields."
    )
    
    print("✅ EXTRACTION SUCCESS!")
    print(f"Provider: {result.provider} / {result.model}")
    print(f"Tokens: {result.input_tokens} + {result.output_tokens} = {result.input_tokens + result.output_tokens}")
    print(f"Processing time: {result.processing_time_ms}ms")
    print(f"Schema valid: {result.is_valid}")
    
    if not result.is_valid:
        print(f"\n⚠️ Validation errors:")
        for error in result.validation_errors:
            print(f"  - {error}")
    
    # Store for later use
    extracted_data_single = result.extracted_data
    
    # Show summary
    print(f"\n📊 Extracted Data:")
    print(f"Bank: {extracted_data_single.get('bank_name', 'N/A')}")
    print(f"Account: {extracted_data_single.get('account_details', {}).get('account_number', 'N/A')}")
    print(f"Transactions: {len(extracted_data_single.get('transactions', []))}")
    
except Exception as e:
    print(f"❌ FAILED: {type(e).__name__}")
    print(f"Error: {str(e)}")

📄 Testing with single page:
Full markdown: 25,779 characters
Single page: 11,538 characters
Reduction: 55.2%

Extracting from single page...

✅ EXTRACTION SUCCESS!
Provider: google / gemini-2.5-flash-lite
Tokens: 4335 + 3164 = 7499
Processing time: 7209ms
Schema valid: True

📊 Extracted Data:
Bank: HSBC
Account: 143-798817-833
Transactions: 32


## Test 6: Alternative Extraction Strategies

The full markdown content may be too large for a single API call. Let's try:
1. Testing with a smaller subset (single page)
2. Using a different model with higher limits
3. Trying OpenAI as an alternative provider

# Test 5: Markdown to JSON Extraction

Test extracting structured JSON from the saved markdown file using the bank statement schema.

In [17]:
# Load the saved markdown file
markdown_file = project_root / "test_output.md"
with open(markdown_file, 'r', encoding='utf-8') as f:
    markdown_content = f.read()

print(f"✅ Loaded markdown file: {markdown_file}")
print(f"Content length: {len(markdown_content)} characters")
print(f"\nFirst 500 characters:\n{markdown_content[:500]}")

# Bank statement schema (from frontend template)
bank_statement_schema = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "title": "Bank Statement Extraction Schema",
    "description": "Schema for extracting structured data from bank statements",
    "type": "object",
    "properties": {
        "bank_name": {
            "type": "string",
            "description": "Name of the financial institution"
        },
        "statement_period": {
            "type": "object",
            "description": "Statement period dates",
            "properties": {
                "start_date": {
                    "type": "string",
                    "format": "date",
                    "description": "Statement start date (YYYY-MM-DD)"
                },
                "end_date": {
                    "type": "string",
                    "format": "date",
                    "description": "Statement end date (YYYY-MM-DD)"
                }
            },
            "required": ["start_date", "end_date"]
        },
        "account_holder": {
            "type": "object",
            "description": "Account holder information",
            "properties": {
                "name": {
                    "type": "string",
                    "description": "Account holder's full name"
                },
                "address": {
                    "type": "string",
                    "description": "Account holder's address"
                }
            },
            "required": ["name"]
        },
        "account_details": {
            "type": "object",
            "description": "Account information",
            "properties": {
                "account_number": {
                    "type": "string",
                    "description": "Bank account number (may be partially masked)"
                },
                "account_type": {
                    "type": "string",
                    "description": "Type of account",
                    "enum": ["checking", "savings", "credit", "investment", "other"]
                },
                "routing_number": {
                    "type": "string",
                    "description": "Bank routing number"
                },
                "currency": {
                    "type": "string",
                    "description": "Account currency code",
                    "pattern": "^[A-Z]{3}$"
                }
            },
            "required": ["account_number", "account_type"]
        },
        "opening_balance": {
            "type": "number",
            "description": "Balance at the beginning of the statement period"
        },
        "closing_balance": {
            "type": "number",
            "description": "Balance at the end of the statement period"
        },
        "transactions": {
            "type": "array",
            "description": "List of transactions during the statement period",
            "items": {
                "type": "object",
                "properties": {
                    "date": {
                        "type": "string",
                        "format": "date",
                        "description": "Transaction date"
                    },
                    "description": {
                        "type": "string",
                        "description": "Transaction description"
                    },
                    "type": {
                        "type": "string",
                        "description": "Transaction type",
                        "enum": ["debit", "credit", "fee", "interest", "transfer", "withdrawal", "deposit", "other"]
                    },
                    "amount": {
                        "type": "number",
                        "description": "Transaction amount (positive for credits, negative for debits)"
                    },
                    "balance": {
                        "type": "number",
                        "description": "Account balance after this transaction"
                    },
                    "reference": {
                        "type": "string",
                        "description": "Transaction reference or check number"
                    },
                    "category": {
                        "type": "string",
                        "description": "Transaction category (e.g., groceries, utilities, salary)"
                    }
                },
                "required": ["date", "description", "amount"]
            }
        },
        "summary": {
            "type": "object",
            "description": "Statement summary",
            "properties": {
                "total_deposits": {
                    "type": "number",
                    "description": "Total amount of deposits/credits",
                    "minimum": 0
                },
                "total_withdrawals": {
                    "type": "number",
                    "description": "Total amount of withdrawals/debits",
                    "minimum": 0
                },
                "total_fees": {
                    "type": "number",
                    "description": "Total fees charged",
                    "minimum": 0
                },
                "interest_earned": {
                    "type": "number",
                    "description": "Interest earned during the period",
                    "minimum": 0
                },
                "transaction_count": {
                    "type": "integer",
                    "description": "Total number of transactions",
                    "minimum": 0
                }
            }
        }
    },
    "required": [
        "bank_name",
        "statement_period",
        "account_holder",
        "account_details",
        "closing_balance",
        "transactions"
    ]
}

print(f"\n✅ Bank statement schema loaded")
print(f"Required fields: {bank_statement_schema['required']}")

✅ Loaded markdown file: /Users/xavierau/Code/python/ai_document_processing/test_output.md
Content length: 25779 characters

First 500 characters:
<!-- PAGE 1 -->
Number 户口号码: 143-798817-833
Branch 分行: MONG KOK

Page 4 of 5
11 December 2024

| Date   | Transaction Details 交易詳情                     | Deposit 存入   | Withdrawal 支出 | Balance 結餘   |
| :----- | :----------------------------------------------- | :------------- | :-------------- | :------------- |
| 6 Dec  | THE HONG KONG JOCKEY                             |                | 880.00          |                |
|        | 35878925AD004686395 06DEC                      |              

✅ Bank statement schema loaded
Required fields: ['bank_name', 'statement_period', 'account_holder', 'account_details', 'closing_balance', 'transactions']


## Schema Validation

Verify the bank statement schema is compatible with Gemini API by removing `$` prefixed metadata fields.

In [26]:
import json

# Clean schema for Gemini compatibility
print("🔍 Schema Validation Check\n")

# Original schema
original_keys = list(bank_statement_schema.keys())
print(f"Original schema keys: {original_keys}")

# Check for $ prefixed fields
metadata_fields = [k for k in original_keys if k.startswith('$')]
if metadata_fields:
    print(f"⚠️  Found metadata fields (will be removed): {metadata_fields}")
else:
    print("✅ No metadata fields found")

# Clean schema using the extractor's method
cleaned_schema = extractor._clean_schema_for_gemini(bank_statement_schema)

# Compare
cleaned_keys = list(cleaned_schema.keys())
print(f"\nCleaned schema keys: {cleaned_keys}")

removed_keys = set(original_keys) - set(cleaned_keys)
if removed_keys:
    print(f"🗑️  Removed keys: {removed_keys}")
else:
    print("✅ No keys removed")

# Show size comparison
original_json = json.dumps(bank_statement_schema, indent=2)
cleaned_json = json.dumps(cleaned_schema, indent=2)

print(f"\n📊 Schema Size:")
print(f"  Original: {len(original_json):,} characters")
print(f"  Cleaned:  {len(cleaned_json):,} characters")
print(f"  Difference: {len(original_json) - len(cleaned_json):,} characters")

# Display cleaned schema
print(f"\n📋 Cleaned Schema Preview (first 500 chars):")
print(cleaned_json)

# Verify required fields are intact
if 'required' in cleaned_schema:
    print(f"\n✅ Required fields preserved: {cleaned_schema['required']}")

print(f"\n✅ Schema is ready for Gemini API")

🔍 Schema Validation Check

Original schema keys: ['$schema', 'title', 'description', 'type', 'properties', 'required']
⚠️  Found metadata fields (will be removed): ['$schema']

Cleaned schema keys: ['title', 'description', 'type', 'properties', 'required']
🗑️  Removed keys: {'$schema'}

📊 Schema Size:
  Original: 4,849 characters
  Cleaned:  4,793 characters
  Difference: 56 characters

📋 Cleaned Schema Preview (first 500 chars):
{
  "title": "Bank Statement Extraction Schema",
  "description": "Schema for extracting structured data from bank statements",
  "type": "object",
  "properties": {
    "bank_name": {
      "type": "string",
      "description": "Name of the financial institution"
    },
    "statement_period": {
      "type": "object",
      "description": "Statement period dates",
      "properties": {
        "start_date": {
          "type": "string",
          "format": "date",
          "description": "Statement start date (YYYY-MM-DD)"
        },
        "end_date": {


## Test 7: Direct Gemini API (No Abstraction)

Test using the official Gemini API directly to isolate if the issue is in our abstraction layer or the API itself.

In [34]:
# Import official Google Generative AI SDK
from  google import genai

# Configure with API key
client = genai.Client(api_key=api_key)

print("✅ Official Gemini API configured")
print(f"API Key: {api_key[:10]}...{api_key[-4:]}")

✅ Official Gemini API configured
API Key: AIzaSyD4Oi...-lu0


In [36]:
# Test 1: Simple example following official docs
# Reference: https://ai.google.dev/gemini-api/docs/structured-output?example=recipe

print("🧪 Test 1: Simple structured output (recipe example)\n")

# Simple schema (like in the docs)
recipe_schema = {
    "type": "object",
    "properties": {
        "recipe_name": {"type": "string"},
        "ingredients": {
            "type": "array",
            "items": {"type": "string"}
        }
    },
    "required": ["recipe_name", "ingredients"]
}

try:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="List a simple chocolate chip cookie recipe.",
        config={
                "response_mime_type":"application/json",
                "response_schema":recipe_schema
            }
        )



    print("✅ SUCCESS!")
    print(f"Response text: {response.text[:200]}...")

    # Parse JSON
    result = json.loads(response.text)
    print(f"\n📊 Parsed result:")
    print(f"  Recipe: {result.get('recipe_name', 'N/A')}")
    print(f"  Ingredients: {len(result.get('ingredients', []))} items")

    # Show usage
    if hasattr(response, 'usage_metadata'):
        print(f"\n💰 Token usage:")
        print(f"  Input: {response.usage_metadata.prompt_token_count}")
        print(f"  Output: {response.usage_metadata.candidates_token_count}")

except Exception as e:
    print(f"❌ FAILED: {type(e).__name__}")
    print(f"Error: {str(e)}")
    import traceback
    traceback.print_exc()

🧪 Test 1: Simple structured output (recipe example)

✅ SUCCESS!
Response text: {"recipe_name":"Simple Chocolate Chip Cookies","ingredients":["1/2 cup (1 stick) unsalted butter, softened","1/2 cup granulated sugar","1/4 cup packed light brown sugar","1 large egg","1 teaspoon vani...

📊 Parsed result:
  Recipe: Simple Chocolate Chip Cookies
  Ingredients: 9 items

💰 Token usage:
  Input: 9
  Output: 83


In [45]:
import time
# Test 2: Bank statement extraction using direct API
print("🧪 Test 2: Bank statement extraction (direct API, single page)\n")

# Use single page first (smaller content)
test_markdown = single_page_markdown

print(f"Content size: {len(test_markdown):,} characters")
print(f"Estimated tokens: ~{len(test_markdown) // 4:,}\n")

# Clean schema (remove $ fields)
cleaned_bank_schema = {k: v for k, v in bank_statement_schema.items() if not k.startswith('$')}

try:
    # Build prompt
    prompt = f"""Extract structured data from this bank statement markdown:

{test_markdown}

Extract all relevant information according to the provided schema. Use null for missing fields."""

    print("Sending request to Gemini API...")
    start_time = time.time()

    # Generate content with structured output

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt,
        config={
                "response_mime_type":"application/json",
                "response_schema":cleaned_bank_schema
            }
        )

    elapsed = time.time() - start_time

    print(f"✅ SUCCESS! ({elapsed:.2f}s)")

    # Parse JSON
    result = json.loads(response.text)

    print(f"\n📊 Extracted Data:")
    print(f"  Bank: {result.get('bank_name', 'N/A')}")
    print(f"  Account: {result.get('account_details', {}).get('account_number', 'N/A')}")
    print(f"  Closing balance: {result.get('closing_balance', 'N/A')}")
    print(f"  Transactions: {len(result.get('transactions', []))}")

    # Show usage
    if hasattr(response, 'usage_metadata'):
        print(f"\n💰 Token usage:")
        print(f"  Input: {response.usage_metadata.prompt_token_count:,}")
        print(f"  Output: {response.usage_metadata.candidates_token_count:,}")
        print(f"  Total: {response.usage_metadata.total_token_count:,}")

    # Store for later
    direct_api_result = result

except Exception as e:
    print(f"❌ FAILED: {type(e).__name__}")
    print(f"Error: {str(e)}")
    import traceback
    traceback.print_exc()

🧪 Test 2: Bank statement extraction (direct API, single page)

Content size: 11,538 characters
Estimated tokens: ~2,884

Sending request to Gemini API...
✅ SUCCESS! (37.29s)

📊 Extracted Data:
  Bank: The Hongkong and Shanghai Banking Corporation Limited
  Account: 143-798817-833
  Closing balance: 59.16
  Transactions: 32

💰 Token usage:
  Input: 3,280
  Output: 4,108
  Total: 7,388


In [46]:
print(result)

{'bank_name': 'The Hongkong and Shanghai Banking Corporation Limited', 'statement_period': {'start_date': '2024-09-01', 'end_date': '2024-11-30'}, 'account_holder': {'name': 'LEE CHUN HEI', 'address': 'null'}, 'account_details': {'account_number': '143-798817-833', 'account_type': 'checking', 'routing_number': 'null', 'currency': 'HKD'}, 'opening_balance': 0.0, 'closing_balance': 59.16, 'transactions': [{'date': '6122-06-06', 'description': 'THE HONG KONG JOCKEY 35878925AD004686395 06DEC LEE C*** H*** 轉賬支出', 'type': 'withdrawal', 'amount': 880.0, 'balance': 0.0, 'reference': '35878925AD004686395', 'category': 'transfer'}, {'date': '6122-06-06', 'description': 'HC124C0673940768 06DEC LEE CHUN HEI 轉賬支出', 'type': 'withdrawal', 'amount': 2000.0, 'balance': 0.0, 'reference': 'HC124C0673940768', 'category': 'transfer'}, {'date': '6122-06-06', 'description': 'HC124C0674070583 06DEC 轉賬收入', 'type': 'deposit', 'amount': 3000.0, 'balance': 0.0, 'reference': 'HC124C0674070583', 'category': 'transf

In [54]:
# Test 3: Full markdown with gemini-2.0-flash (direct API)
print("🧪 Test 3: Full bank statement (all pages, direct API)\n")

print(f"Content size: {len(markdown_content):,} characters")
print(f"Estimated tokens: ~{len(markdown_content) // 4:,}")
print(f"Model: gemini-2.0-flash (not lite)\n")

try:
    # Build prompt
    prompt = f"""Extract structured data from this multi-page bank statement markdown:

{markdown_content}

Extract all relevant information from ALL pages. Combine transactions from all pages into a single array. Use null for missing fields."""

    print("Sending request to Gemini API...")
    start_time = time.time()

    # Generate content with structured output
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "response_mime_type":"application/json",
            "response_schema":cleaned_bank_schema,
            "temperature":0
        }
    )

    elapsed = time.time() - start_time

    print(f"✅ SUCCESS! ({elapsed:.2f}s)")

    # Parse JSON
    result = json.loads(response.text)

    print(f"\n📊 Extracted Data:")
    print(f"  Bank: {result.get('bank_name', 'N/A')}")
    print(f"  Account: {result.get('account_details', {}).get('account_number', 'N/A')}")
    print(f"  Period: {result.get('statement_period', {}).get('start_date', 'N/A')} to {result.get('statement_period', {}).get('end_date', 'N/A')}")
    print(f"  Closing balance: {result.get('closing_balance', 'N/A')}")
    print(f"  Transactions: {len(result.get('transactions', []))}")

    # Show usage
    if hasattr(response, 'usage_metadata'):
        print(f"\n💰 Token usage:")
        print(f"  Input: {response.usage_metadata.prompt_token_count:,}")
        print(f"  Output: {response.usage_metadata.candidates_token_count:,}")
        print(f"  Total: {response.usage_metadata.total_token_count:,}")

    # Store for later
    extracted_data = result

    print("\n✅ Full document extraction complete!")

except Exception as e:
    print(f"❌ FAILED: {type(e).__name__}")
    print(f"Error: {str(e)}")
    import traceback
    traceback.print_exc()

    print("\n💡 If this fails, the content may be too large even for the full model.")
    print("   Consider chunking the markdown into smaller pieces.")

🧪 Test 3: Full bank statement (all pages, direct API)

Content size: 25,779 characters
Estimated tokens: ~6,444
Model: gemini-2.0-flash (not lite)

Sending request to Gemini API...
❌ FAILED: RemoteProtocolError
Error: Server disconnected without sending a response.

💡 If this fails, the content may be too large even for the full model.
   Consider chunking the markdown into smaller pieces.


Traceback (most recent call last):
  File "/Users/xavierau/Code/python/ai_document_processing/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/Users/xavierau/Code/python/ai_document_processing/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/xavierau/Code/python/ai_document_processing/.venv/lib/python3.12/site-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/Users/xavierau/Code/python/ai_document_processing/.venv/lib/python3.12/site-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/xavierau/Code/python/ai_document_processing/.venv/lib/python3.12/site-packages/httpcore/_sync/connection.py", line 10

## Direct API Test Summary

These tests use the **official Google Generative AI SDK** (`google.generativeai`) directly with zero abstraction layers.

**Test Progression**:
1. ✅ **Simple Recipe**: Verify basic API functionality
2. ✅ **Single Page**: Test with bank statement (smaller content)
3. ✅ **Full Document**: Test with all 4 pages (~28KB markdown)

**Key Differences from Abstraction Layer**:
- Uses `genai.GenerativeModel()` directly (official SDK)
- Uses `genai.GenerationConfig()` for structured output
- No custom client wrappers or middleware
- Follows official docs pattern exactly

**Expected Results**:
- Test 1 should always succeed (simple example)
- Test 2 should succeed (single page ~7KB)
- Test 3 may fail if content exceeds limits

If all tests succeed, the issue is in our abstraction layer. If tests fail, it's an API limitation.

In [27]:
# Import the markdown JSON extractor
from app.services.converters.markdown_json_extractor import MarkdownJsonExtractor

# Initialize the extractor with Gemini
extractor = MarkdownJsonExtractor(
    provider="google",
    model="gemini-2.5-flash-lite",
    api_key=api_key
)

print("✅ MarkdownJsonExtractor initialized")
print(f"Provider: {extractor.provider}")
print(f"Model: {extractor.model}")

✅ MarkdownJsonExtractor initialized
Provider: google
Model: gemini-2.5-flash-lite


In [28]:
# Extract structured JSON from markdown
print("Extracting structured JSON from markdown...\n")

try:
    result = await extractor.extract(
        markdown_content=markdown_content,
        schema=bank_statement_schema,
        custom_prompt="Extract all bank statement information including all transactions accurately."
    )

    print("✅ EXTRACTION SUCCESS!")
    print(f"\nProvider: {result.provider}")
    print(f"Model: {result.model}")
    print(f"Input tokens: {result.input_tokens}")
    print(f"Output tokens: {result.output_tokens}")
    print(f"Total tokens: {result.input_tokens + result.output_tokens}")
    print(f"Processing time: {result.processing_time_ms}ms")
    print(f"Schema valid: {result.is_valid}")

    if not result.is_valid:
        print(f"\n⚠️ Validation errors:")
        for error in result.validation_errors:
            print(f"  - {error}")

    # Store for later use
    extracted_data = result.extracted_data

    # Show summary
    print(f"\n📊 Extracted Data Summary:")
    print(f"Bank: {extracted_data.get('bank_name', 'N/A')}")
    print(f"Account: {extracted_data.get('account_details', {}).get('account_number', 'N/A')}")
    print(f"Period: {extracted_data.get('statement_period', {}).get('start_date', 'N/A')} to {extracted_data.get('statement_period', {}).get('end_date', 'N/A')}")
    print(f"Closing balance: {extracted_data.get('closing_balance', 'N/A')}")
    print(f"Transactions: {len(extracted_data.get('transactions', []))}")

except Exception as e:
    print(f"❌ EXTRACTION FAILED: {type(e).__name__}")
    print(f"Error: {str(e)}")
    import traceback
    traceback.print_exc()

Extracting structured JSON from markdown...



Markdown-to-JSON extraction failed: Server disconnected without sending a response.


❌ EXTRACTION FAILED: Exception
Error: Markdown-to-JSON extraction error: Server disconnected without sending a response.


Traceback (most recent call last):
  File "/Users/xavierau/Code/python/ai_document_processing/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/Users/xavierau/Code/python/ai_document_processing/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/xavierau/Code/python/ai_document_processing/.venv/lib/python3.12/site-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/Users/xavierau/Code/python/ai_document_processing/.venv/lib/python3.12/site-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/xavierau/Code/python/ai_document_processing/.venv/lib/python3.12/site-packages/httpcore/_sync/connection.py", line 10

In [ ]:
# Display sample transactions
if 'extracted_data' in locals() and 'transactions' in extracted_data:
    transactions = extracted_data['transactions']

    print(f"📋 Sample Transactions (showing first 5 of {len(transactions)}):\n")

    for i, txn in enumerate(transactions[:5], 1):
        print(f"{i}. Date: {txn.get('date', 'N/A')}")
        print(f"   Description: {txn.get('description', 'N/A')[:60]}...")
        print(f"   Type: {txn.get('type', 'N/A')}")
        print(f"   Amount: {txn.get('amount', 'N/A')}")
        if 'balance' in txn:
            print(f"   Balance: {txn['balance']}")
        print()

    # Show summary if available
    if 'summary' in extracted_data:
        summary = extracted_data['summary']
        print(f"💰 Statement Summary:")
        print(f"  Total deposits: {summary.get('total_deposits', 'N/A')}")
        print(f"  Total withdrawals: {summary.get('total_withdrawals', 'N/A')}")
        print(f"  Total fees: {summary.get('total_fees', 'N/A')}")
        print(f"  Interest earned: {summary.get('interest_earned', 'N/A')}")
        print(f"  Transaction count: {summary.get('transaction_count', 'N/A')}")
else:
    print("❌ No extracted data available")

In [ ]:
# Save extracted JSON to file
if 'extracted_data' in locals():
    output_file = project_root / "test_extracted_data.json"

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(extracted_data, f, indent=2, ensure_ascii=False)

    print(f"✅ Saved extracted JSON to: {output_file}")
    print(f"File size: {output_file.stat().st_size:,} bytes")

    # Show file stats
    print(f"\n📈 Extraction Statistics:")
    print(f"  Markdown input: {len(markdown_content):,} characters")
    print(f"  JSON output: {output_file.stat().st_size:,} bytes")
    print(f"  Compression ratio: {len(markdown_content) / output_file.stat().st_size:.2f}x")
    print(f"  Tokens used: {result.input_tokens + result.output_tokens:,}")
    print(f"  Processing time: {result.processing_time_ms:,}ms")
else:
    print("❌ No extracted data to save")

## Summary

This notebook demonstrates the complete markdown pipeline:

1. **Image → Markdown Conversion**:
   - Used `GeminiMarkdownConverter` to convert bank statement images to markdown
   - Processed 4 pages in batch mode
   - Generated structured markdown with table formatting

2. **Markdown → JSON Extraction**:
   - Used `MarkdownJsonExtractor` to extract structured data from markdown
   - Applied bank statement JSON schema for validation
   - Extracted account details, transactions, and summary information

**Key Benefits of Markdown Pipeline**:
- ✅ **Cost Effective**: Text-only models (gemini-2.0-flash-lite) are 10x cheaper than vision models
- ✅ **Accurate**: Markdown provides clear structure for LLM to parse
- ✅ **Flexible**: Can regenerate JSON with different schemas without re-processing images
- ✅ **Debuggable**: Human-readable markdown intermediate format

**Files Generated**:
- `test_output.md` - Markdown conversion of bank statement
- `test_extracted_data.json` - Structured JSON extraction